# FAC_NB_Spark_vs_TSQL — la misma agregación en Spark SQL y en T-SQL

**Pregunta de negocio:** por mes, provincia (de la sucursal) y canal, ¿cuántas transacciones, cuánto crédito,
cuánto débito y cuál es el ticket promedio? Une transacciones → cuentas → clientes → sucursales.

| Variante | Motor | Origen |
|---|---|---|
| 1 | Spark SQL (este notebook) | Tablas plata de `AB_Lakehouse` (Delta, solo lectura) |
| 2 | T-SQL en el Warehouse | `FAC_WH_Gold` (modelo estrella cargado por procedimientos) |
| 3 (opcional) | T-SQL en el SQL analytics endpoint del Lakehouse | Las mismas tablas plata de la variante 1 |

**Sobre la medición.** Cada variante se repite; la primera repetición se muestra aparte porque suele incluir
metadatos y lectura en frío. En las variantes 2 y 3 el tiempo medido aquí incluye red, conversión y
transferencia del conector de Spark para Warehouse; el tiempo del servidor está en
`queryinsights.exec_requests_history` (ver `comparacion_tsql.sql`). **No se promete ningún resultado de
rendimiento**: lo que sí debe coincidir son los resultados.

In [ ]:
p_lakehouse_origen = "AB_Lakehouse"
p_warehouse = "FAC_WH_Gold"
p_repeticiones = 3
p_medir_endpoint = True

## Paso 1 — Funciones comunes (`%run`)

In [ ]:
%run FAC_NB_Utils

## Paso 2 — Registrar las tablas plata como vistas temporales de Spark

In [ ]:
import json
import statistics
import time
from decimal import Decimal

for tabla in ("ab_silver_transacciones", "ab_silver_cuentas", "ab_silver_clientes", "ab_silver_sucursales"):
    leer_tabla_delta(spark, p_lakehouse_origen, tabla).createOrReplaceTempView(tabla)

print("filas en ab_silver_transacciones:", spark.sql("SELECT COUNT(*) AS filas FROM ab_silver_transacciones").first()["filas"])  # esperado: 72000

In [ ]:
def medir(nombre, ejecutar, repeticiones):
    """Ejecuta `ejecutar` varias veces y devuelve los segundos de cada repetición y las filas."""
    segundos = []
    filas = None
    for _ in range(int(repeticiones)):
        t0 = time.perf_counter()
        filas = ejecutar()
        segundos.append(round(time.perf_counter() - t0, 3))
    return {"variante": nombre, "segundos": segundos, "filas": len(filas)}, filas


def clave(fila):
    return (fila["mes"], fila["provincia"], fila["canal"])

## Paso 3 — Variante 1: Spark SQL

In [ ]:
SQL_SPARK = """
SELECT date_format(t.fecha, 'yyyy-MM') AS mes,
       s.provincia,
       t.canal,
       COUNT(*) AS transacciones,
       SUM(CASE WHEN t.direccion = 'CREDITO' THEN t.monto ELSE 0 END) AS monto_creditos,
       SUM(CASE WHEN t.direccion = 'DEBITO'  THEN t.monto ELSE 0 END) AS monto_debitos,
       ROUND(AVG(t.monto), 2) AS ticket_promedio
FROM ab_silver_transacciones t
JOIN ab_silver_cuentas    c  ON c.cuenta_id   = t.cuenta_id
JOIN ab_silver_clientes   cl ON cl.cliente_id = c.cliente_id
JOIN ab_silver_sucursales s  ON s.sucursal_id = cl.sucursal_id
GROUP BY date_format(t.fecha, 'yyyy-MM'), s.provincia, t.canal
ORDER BY mes, s.provincia, t.canal
"""

res_spark, filas_spark = medir("Spark SQL (tablas plata del Lakehouse)",
                               lambda: spark.sql(SQL_SPARK).collect(), p_repeticiones)
print(res_spark)

## Paso 4 — Variante 2: T-SQL en el Warehouse

Usa el conector de Spark para Fabric Warehouse con una consulta de paso (*passthrough*): el texto T-SQL se
ejecuta en el motor SQL y el resultado vuelve a Spark. Solo admite autenticación interactiva de usuario.

In [ ]:
import com.microsoft.spark.fabric
from com.microsoft.spark.fabric.Constants import Constants

SQL_TSQL_WH = """
/* FAC_CMP_WH_conector */
SELECT m.mes, s.provincia, m.canal,
       COUNT_BIG(*) AS transacciones,
       SUM(CASE WHEN m.direccion = 'CREDITO' THEN m.monto ELSE CAST(0 AS decimal(18,2)) END) AS monto_creditos,
       SUM(CASE WHEN m.direccion = 'DEBITO'  THEN m.monto ELSE CAST(0 AS decimal(18,2)) END) AS monto_debitos,
       CAST(AVG(m.monto) AS decimal(18,2)) AS ticket_promedio
FROM (SELECT CONCAT(CAST(YEAR(t.fecha) AS varchar(4)), '-', RIGHT(CONCAT('0', CAST(MONTH(t.fecha) AS varchar(2))), 2)) AS mes,
             t.cuenta_id, t.canal, t.direccion, t.monto
      FROM transacciones.fact_transaccion AS t) AS m
JOIN cuentas.dim_cuenta    AS c  ON c.cuenta_id   = m.cuenta_id
JOIN clientes.dim_cliente   AS cl ON cl.cliente_id = c.cliente_id
JOIN clientes.dim_sucursal AS s  ON s.sucursal_id = cl.sucursal_id
GROUP BY m.mes, s.provincia, m.canal
"""

res_wh, filas_wh = medir("T-SQL Warehouse FAC_WH_Gold (conector)",
                         lambda: spark.read.option(Constants.DatabaseName, p_warehouse)
                                      .synapsesql(SQL_TSQL_WH).collect(), p_repeticiones)
print(res_wh)

## Paso 5 (opcional) — Variante 3: T-SQL en el SQL analytics endpoint del Lakehouse

In [ ]:
SQL_TSQL_EP = """
/* FAC_CMP_EP_conector */
SELECT m.mes, s.provincia, m.canal,
       COUNT_BIG(*) AS transacciones,
       SUM(CASE WHEN m.direccion = 'CREDITO' THEN m.monto ELSE CAST(0 AS decimal(18,2)) END) AS monto_creditos,
       SUM(CASE WHEN m.direccion = 'DEBITO'  THEN m.monto ELSE CAST(0 AS decimal(18,2)) END) AS monto_debitos,
       CAST(AVG(m.monto) AS decimal(18,2)) AS ticket_promedio
FROM (SELECT CONCAT(CAST(YEAR(t.fecha) AS varchar(4)), '-', RIGHT(CONCAT('0', CAST(MONTH(t.fecha) AS varchar(2))), 2)) AS mes,
             t.cuenta_id, t.canal, t.direccion, t.monto
      FROM dbo.ab_silver_transacciones AS t) AS m
JOIN dbo.ab_silver_cuentas    AS c  ON c.cuenta_id   = m.cuenta_id
JOIN dbo.ab_silver_clientes   AS cl ON cl.cliente_id = c.cliente_id
JOIN dbo.ab_silver_sucursales AS s  ON s.sucursal_id = cl.sucursal_id
GROUP BY m.mes, s.provincia, m.canal
"""

if p_medir_endpoint:
    res_ep, filas_ep = medir("T-SQL endpoint SQL del Lakehouse (conector)",
                             lambda: spark.read.option(Constants.DatabaseName, p_lakehouse_origen)
                                          .synapsesql(SQL_TSQL_EP).collect(), p_repeticiones)
    print(res_ep)
else:
    res_ep, filas_ep = None, None

## Paso 6 — Los resultados deben coincidir

Se compara cada grupo (mes, provincia, canal) en transacciones y montos exactos. El ticket promedio queda fuera
de la igualdad porque el redondeo del promedio puede diferir entre motores.

In [ ]:
ESPERADO = {
    "grupos": 210,
    "transacciones": 72000,
    "monto_creditos": Decimal("14976153627.79"),
    "monto_debitos": Decimal("4530373831.90"),
}


def firma(fila):
    return (int(fila["transacciones"]), Decimal(fila["monto_creditos"]), Decimal(fila["monto_debitos"]))


def resumen_totales(filas):
    return {
        "grupos": len(filas),
        "transacciones": sum(int(f["transacciones"]) for f in filas),
        "monto_creditos": sum((Decimal(f["monto_creditos"]) for f in filas), Decimal("0")),
        "monto_debitos": sum((Decimal(f["monto_debitos"]) for f in filas), Decimal("0")),
    }


motores = {"spark": filas_spark, "warehouse": filas_wh}
if filas_ep is not None:
    motores["endpoint"] = filas_ep

mapas = {nombre: {clave(f): firma(f) for f in filas} for nombre, filas in motores.items()}
for nombre, filas in motores.items():
    totales = resumen_totales(filas)
    print(f"{nombre:10s}", totales, "| coincide con el laboratorio:", totales == ESPERADO)

iguales = all(mapas[n] == mapas["spark"] for n in mapas)
print("Todos los motores devuelven exactamente los mismos grupos y montos:", iguales)
assert iguales, "Los motores no coinciden: revise la carga del Warehouse (operacion.usp_reconciliar)"

## Paso 7 — Tabla de tiempos (sin conclusiones automáticas)

In [ ]:
resultados = [r for r in (res_spark, res_wh, res_ep) if r]
for r in resultados:
    resto = r["segundos"][1:]
    print(f"{r['variante']:55s} primera: {r['segundos'][0]:7.3f} s | resto (mediana): "
          f"{(statistics.median(resto) if resto else float('nan')):7.3f} s | repeticiones: {r['segundos']}")

escribir_bitacora(spark, "spark_vs_tsql", "OK", filas=res_spark["filas"],
                  mensaje=json.dumps({"motores_iguales": iguales, "resultados": resultados}, ensure_ascii=False))
display(leer_bitacora(spark, ultimas=3))

## Cómo leer el resultado en la demo

- Compare **primera repetición** frente a las siguientes: muestra el efecto de lo frío frente a lo tibio, no de un motor.
- Los tiempos de las variantes 2 y 3 incluyen la transferencia del conector; para el tiempo del servidor use
  `queryinsights.exec_requests_history` con la etiqueta `FAC_CMP_*` (puede tardar hasta 15 minutos en aparecer).
- Cambie `p_repeticiones` y vuelva a ejecutar: conserve las cifras que mida, sin extrapolarlas a otros volúmenes.
- Mismo resultado, distinta herramienta: Spark escala horizontalmente y escribe Delta; T-SQL en Warehouse da
  transacciones entre tablas, procedimientos y seguridad en el motor SQL. Elija por el equipo y la carga, no por una cifra.